<a href="https://colab.research.google.com/github/Santy-Ruiz/WebScraping_Portatiles_Exito/blob/main/EA2_Cabana_Marian_Ruiz_Santiago.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install google-colab-selenium --quiet

import google_colab_selenium as gs
from bs4 import BeautifulSoup
import pandas as pd
import time
from google.colab import files


In [4]:
print("Levantando el navegador fantasma... esto puede tomar unos segundos.")
driver = gs.Chrome()

# Lista global para consolidar los productos de todas las páginas
todos_los_portatiles = []

# Definimos cuántas páginas queremos descargar
paginas_a_descargar = 4

print("\nIniciando la extracción multicanal con paginación...")

for pagina in range(1, paginas_a_descargar + 1):
    url_dinamica = f"https://www.exito.com/tecnologia/computadores/portatiles?category-1=tecnologia&category-2=computadores&category-3=portatiles&facets=category-1%2Ccategory-2%2Ccategory-3&sort=score_desc&page={pagina}"

    print(f"Procesando página {pagina} de {paginas_a_descargar}... ", end="")

    try:
        driver.get(url_dinamica)
        # Le damos 5 segundos a la pagina para que termine de renderizar el JavaScript y los precios
        time.sleep(5)

        # Pasamos el HTML ya renderizado a BeautifulSoup
        soup_pag = BeautifulSoup(driver.page_source, 'html.parser')

        # Buscamos el contenedor usando lambda para ignorar los códigos alfanuméricos variables
        contenedores = soup_pag.find_all("div", class_=lambda x: x and "productCard_productInfo" in x)

        conteo_pagina = 0
        for caja in contenedores:
            # Nombre y Marca
            nombre_tag = caja.find("a", {"data-testid": "product-link"})
            nombre = nombre_tag.text.strip() if nombre_tag else "No disponible"

            marca_tag = caja.find("h3", class_=lambda x: x and "styles_brand" in x)
            marca = marca_tag.text.strip() if marca_tag else "Genérica"

            # Precio 1: Otros medios de pago
            precio_otros_tag = caja.find("p", {"data-fs-container-price-otros": "true"})
            precio_otros = precio_otros_tag.text.strip() if precio_otros_tag else "Consultar"

            # Precio 2: Tarjeta Aliada (Éxito)
            precio_aliado_tag = caja.find("p", class_=lambda x: x and "allieds-display_bestDiscount__Kb8eD" in x)
            precio_aliado = precio_aliado_tag.text.replace("price", "").strip() if precio_aliado_tag else "No aplica"

            # Descuento general
            descuento_tag = caja.find("div", class_=lambda x: x and "priceSection_container-promotion_discount" in x)
            descuento = descuento_tag.text.strip() if descuento_tag else "0%"

            todos_los_portatiles.append({
                "Marca": marca,
                "Computador": nombre,
                "Precio (Otros medios)": precio_otros,
                "Precio (Tarjeta Aliada)": precio_aliado,
                "Descuento": descuento,
                "Ubicación": f"Pág {pagina}"
            })
            conteo_pagina += 1

        print(f"¡Completada! Se encontraron {conteo_pagina} productos.")

    except Exception as e:
        print(f"Error al procesar la página {pagina}: {str(e)}")

driver.quit()

# Creamos el DataFrame consolidador
df_exito = pd.DataFrame(todos_los_portatiles)

if not df_exito.empty:
   # Eliminamos posibles registros duplicados
    df_exito = df_exito.drop_duplicates(subset=['Computador'])

    print(f"\nExtracción finalizada. Total de productos únicos recolectados: {len(df_exito)}")
    display(df_exito.head(15))

    # Exportamos y descargamos automáticamente el archivo
    nombre_archivo = "portatiles_exito_doble_precio.csv"
    df_exito.to_csv(nombre_archivo, index=False, encoding='utf-8-sig', sep=';')
    files.download(nombre_archivo)
else:
    print("\nNo se extrajeron datos.")

Levantando el navegador fantasma... esto puede tomar unos segundos.


<IPython.core.display.Javascript object>


Iniciando la extracción multicanal con paginación...
Procesando página 1 de 4... ¡Completada! Se encontraron 16 productos.
Procesando página 2 de 4... ¡Completada! Se encontraron 16 productos.
Procesando página 3 de 4... ¡Completada! Se encontraron 14 productos.
Procesando página 4 de 4... ¡Completada! Se encontraron 13 productos.

Extracción finalizada. Total de productos únicos recolectados: 59


,Marca,Computador,Precio (Otros medios),Precio (Tarjeta Aliada),Descuento,Ubicación
0,HP,Portátil HP AMD Ryzen 7 7730U RAM 16 GB 512 G...,$ 2.618.000,$ 2.392.255,-47%,Pág 1
1,HP,Portátil HP OmniBook 3 Intel Core 5 120U RAM 8...,$ 2.199.000,$ 2.021.604,-54%,Pág 1
2,ASUS,Portátil ASUS Vivobook Go 15 AMD Ryzen 3 7320U...,$ 1.682.949,$ 1.583.952,-52%,Pág 1
3,HP,Portátil Hp Victus 15 Fb3019la Ryzen 7 7445Hs ...,$ 3.299.990,$ 3.167.990,-53%,Pág 1
4,ACER,Portátil ACER Aspire Go AMD Ryzen 7 5825U RAM ...,$ 2.464.000,$ 2.310.000,-40%,Pág 1
5,ASUS,Portátil ASUS Vivobook 15 Intel Core 5 120U RA...,$ 2.549.949,$ 2.199.900,-56%,Pág 1
6,LENOVO,Portátil Lenovo Ideapad Slim 3 15Iru8 | Core I...,$ 1.530.000,No aplica,0%,Pág 1
7,ASUS,Portátil ASUS Vivobook 16 Intel Core i5 13420H...,$ 3.999.900,No aplica,0%,Pág 1
8,ACER,Portátil ACER Aspire L Intel Core i7 1355U RAM...,$ 2.618.000,$ 2.460.920,-53%,Pág 1
9,HP,Portátil Hp Intel Dual Core N4500 Ssd 512Gb Ra...,$ 1.067.845,No aplica,-37%,Pág 1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>